# 04 — Graph: Hospital Network

**Purpose:** Model WA hospital-to-health-service relationships as a graph.  
Hospitals in the same health service are connected — enabling network-level analysis.  

**Source:** `silver.dim_hospitals`  
**Targets:**
- `gold.hospital_network_edges` (Delta table — graph edges)
- `gold.hospital_network_nodes` (Delta table — graph nodes)

This mirrors the Neo4j graph approach used at UWA for mining incident analysis,
applied here to health service network modelling.

In [ ]:
# ------------------------------------------------------------------
# 1. Load hospital dimension
# ------------------------------------------------------------------
from pyspark.sql.functions import col, count, collect_set, size

hospitals = spark.table("silver.dim_hospitals")
print(f"Total hospitals: {hospitals.count()}")
hospitals.show(5)

In [ ]:
# ------------------------------------------------------------------
# 2. Build graph nodes
#    Node types: Hospital, HealthService
# ------------------------------------------------------------------
from pyspark.sql.functions import lit, monotonically_increasing_id

# Hospital nodes
hospital_nodes = hospitals.select(
    col("site_id").alias("node_id"),
    col("hospital_name").alias("name"),
    lit("Hospital").alias("node_type"),
    col("facility_type"),
    col("suburb"),
    col("latitude"),
    col("longitude")
)

# Health service nodes
service_nodes = hospitals \
    .select("health_service") \
    .distinct() \
    .filter(col("health_service").isNotNull()) \
    .select(
        col("health_service").alias("node_id"),
        col("health_service").alias("name"),
        lit("HealthService").alias("node_type"),
        lit(None).cast("string").alias("facility_type"),
        lit(None).cast("string").alias("suburb"),
        lit(None).cast("double").alias("latitude"),
        lit(None).cast("double").alias("longitude")
    )

all_nodes = hospital_nodes.unionByName(service_nodes)
print(f"Total graph nodes: {all_nodes.count()}")
all_nodes.groupBy("node_type").count().show()

In [ ]:
# ------------------------------------------------------------------
# 3. Build graph edges
#    Edge: (Hospital) -[BELONGS_TO]-> (HealthService)
#    Edge: (Hospital) -[CO_LOCATED_IN]-> (Hospital)  [same suburb]
# ------------------------------------------------------------------

# Edge type 1: Hospital BELONGS_TO HealthService
belongs_to_edges = hospitals \
    .filter(col("health_service").isNotNull()) \
    .select(
        col("site_id").alias("src_node_id"),
        col("hospital_name").alias("src_name"),
        col("health_service").alias("dst_node_id"),
        col("health_service").alias("dst_name"),
        lit("BELONGS_TO").alias("relationship")
    )

# Edge type 2: Hospital CO_LOCATED_IN (same health service) — peer connections
h1 = hospitals.alias("h1")
h2 = hospitals.alias("h2")

peer_edges = h1.join(
    h2,
    (col("h1.health_service") == col("h2.health_service")) &
    (col("h1.site_id") != col("h2.site_id")),
    "inner"
).select(
    col("h1.site_id").alias("src_node_id"),
    col("h1.hospital_name").alias("src_name"),
    col("h2.site_id").alias("dst_node_id"),
    col("h2.hospital_name").alias("dst_name"),
    lit("PEER_IN_SERVICE").alias("relationship")
).distinct()

all_edges = belongs_to_edges.unionByName(peer_edges)
print(f"Total graph edges: {all_edges.count()}")
all_edges.groupBy("relationship").count().show()

In [ ]:
# ------------------------------------------------------------------
# 4. Enrich edges with ED performance context
#    Join gold layer to annotate which hospitals are below target
# ------------------------------------------------------------------
from pyspark.sql.functions import max, avg as spark_avg, round

# Latest period performance per hospital
latest_perf = spark.table("gold.ed_waittime_trends") \
    .groupBy("hospital_code", "hospital_name") \
    .agg(
        round(spark_avg("four_hour_departure_rate"), 2).alias("avg_4hr_rate"),
        max("below_target").alias("ever_below_target")
    )

# Annotate hospital nodes with performance
enriched_nodes = all_nodes \
    .filter(col("node_type") == "Hospital") \
    .join(latest_perf, all_nodes.name == latest_perf.hospital_name, "left") \
    .drop("hospital_code", "hospital_name") \
    .unionByName(
        all_nodes.filter(col("node_type") == "HealthService")
            .withColumn("avg_4hr_rate", lit(None).cast("double"))
            .withColumn("ever_below_target", lit(None).cast("boolean"))
    )

print("Sample enriched hospital nodes:")
enriched_nodes.filter(col("node_type") == "Hospital").show(10)

In [ ]:
# ------------------------------------------------------------------
# 5. Write gold Delta tables
# ------------------------------------------------------------------
all_edges.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.hospital_network_edges")

enriched_nodes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.hospital_network_nodes")

print(f"gold.hospital_network_edges: {spark.table('gold.hospital_network_edges').count()} rows")
print(f"gold.hospital_network_nodes: {spark.table('gold.hospital_network_nodes').count()} rows")
print("\nGraph layer written successfully")

## Neo4j Cypher (Option B)

If loading into a local Neo4j instance, use these Cypher queries:

```cypher
// Create hospital nodes
LOAD CSV WITH HEADERS FROM 'file:///hospital_nodes.csv' AS row
CREATE (:Hospital {
  site_id: row.node_id,
  name: row.name,
  facility_type: row.facility_type,
  suburb: row.suburb,
  avg_4hr_rate: toFloat(row.avg_4hr_rate)
});

// Create health service nodes
LOAD CSV WITH HEADERS FROM 'file:///service_nodes.csv' AS row
CREATE (:HealthService { name: row.name });

// Create BELONGS_TO relationships
LOAD CSV WITH HEADERS FROM 'file:///edges.csv' AS row
MATCH (h:Hospital {site_id: row.src_node_id})
MATCH (s:HealthService {name: row.dst_name})
WHERE row.relationship = 'BELONGS_TO'
CREATE (h)-[:BELONGS_TO]->(s);

// Query: which health services have the most underperforming hospitals?
MATCH (h:Hospital)-[:BELONGS_TO]->(s:HealthService)
WHERE h.avg_4hr_rate < 67
RETURN s.name AS health_service,
       count(h) AS underperforming_hospitals,
       avg(h.avg_4hr_rate) AS avg_rate
ORDER BY underperforming_hospitals DESC;
```